In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
import pycountry
from matplotlib.pyplot import cm
from os import listdir as ls

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import get_all_like_comps, get_param_vals_by_analysis, get_completed_chains

In [ ]:
import arviz as az
from emu_renewal.outputs import get_country_analyses

In [ ]:
colours = cm.Dark2.colors

In [ ]:
job_path = OUTPUTS_PATH / "42780561"
countries = ls(job_path)
countries = countries[:2]

In [ ]:
get_completed_chains(job_path)

In [ ]:
proc_disp_samples = {}
fit_disp_samples = {}
like_comps = {}
for country in countries:
    country_path = job_path / country
    proc_disp_samples[country] = get_param_vals_by_analysis("dispersion_proc", country_path)
    fit_disp_samples[country] = get_param_vals_by_analysis("shared_dispersion", country_path)
    like_comps[country] = get_all_like_comps(country_path)

In [ ]:
def plot_country_like_comps(country, like_comps, fit_disp_samples, proc_disp_samples):
    c_like_comps = like_comps[country].swaplevel(axis=1)
    c_fit_disps = fit_disp_samples[country]
    c_proc_disps = proc_disp_samples[country]
    comp_fig, axes = plt.subplots(4, 2, figsize=[11, 14])
    country_name = pycountry.countries.lookup(country).name
    comp_fig.suptitle(country_name, fontsize=15)
    flat_axes = axes.ravel()
    for t, targ in enumerate(set(c_like_comps.columns.get_level_values(0))):
        t_ax = flat_axes[t]
        t_ax = sns.kdeplot(c_like_comps[targ], fill=True, ax=t_ax)
        t_ax.set_title(targ)
        flat_axes[t].get_legend().remove()
    for t_ax in flat_axes:
        t_ax.set_yticks([])
        t_ax.set_ylabel("")
    sns.kdeplot(c_fit_disps, fill=True, ax=flat_axes[-2])
    flat_axes[-2].get_legend().remove()
    flat_axes[-2].set_title("shared_dispersion")   
    sns.kdeplot(c_proc_disps, fill=True, ax=flat_axes[-1])
    flat_axes[-1].set_title("proc_dispersion")
    comp_fig.tight_layout()

In [ ]:
for country in countries:
    plot_country_like_comps(country, like_comps, fit_disp_samples, proc_disp_samples)